In [2]:
##### import sys
import time
import multiprocessing
from reset import SystemCleanup
from ingestion.producer import EventProducer
from ingestion.consumer import EventConsumer
from processing.spark_stream_processor import SparkStructuredStreaming
from processing.batch_processor import BatchProcessor
from storage.mongo_loader import MongoLoader
from storage.neo4j_loader import Neo4jLoader
from storage.mongo_queries import MongoQueries
from storage.neo4j_queries import Neo4jQueries
 
def run_streamer():
    streamer = SparkStructuredStreaming()
    streamer.start_streaming()

def run_consumer():
    consumer = EventConsumer()
    consumer.run()

def run_producer():
    publisher = EventProducer()
    publisher.run()

class MainMenu:
    def __init__(self):
        self.running = True

    def display_menu(self):
        print("\n" + "="*40)
        print("    CENTRAL KITCHEN DATA PIPELINE")
        print("="*40)
        print("1. Run Full Pipeline (Produce Data & Load DBs)")
        print("2. Run Batch Processing (Spark Reports)")
        print("3. Run MongoDB Queries")
        print("4. Run Neo4j Queries")
        print("5. Reset System (Clear Kafka, DBs, HDFS)")
        print("0. Exit")
        print("="*40)

    def run_full_pipeline(self):
        print("\nStarting the automated data pipeline.")
        
        stream_process = multiprocessing.Process(target=run_streamer)
        consumer_process = multiprocessing.Process(target=run_consumer)
        producer_process = multiprocessing.Process(target=run_producer)

        print("Starting Spark Stream Processor...")
        stream_process.start()
        time.sleep(10) 

        print("Starting Kafka Consumer...")
        consumer_process.start()
        time.sleep(5) 

        print("Starting Kafka Producer...")
        producer_process.start()
        
        producer_process.join()
        print("Producer finished generating data.")

        print("Waiting for final events to process...")
        time.sleep(15)
        
        print("Stopping Stream Processor and Consumer...")
        stream_process.terminate()
        consumer_process.terminate()
        stream_process.join()
        consumer_process.join()

        print("Loading data into MongoDB...")
        mongo_loader = MongoLoader()
        mongo_loader.run_transfer()

        print("Loading data into Neo4j...")
        neo4j_loader = Neo4jLoader()
        neo4j_loader.run_transfer()

        print("Full pipeline execution complete.")

    def run(self):
        while self.running:
            self.display_menu()
            choice = input("Enter your choice (0-5): ").strip()

            if choice == '1':
                self.run_full_pipeline()
            elif choice == '2':
                processor = BatchProcessor()
                processor.run_tests()
            elif choice == '3':
                queries = MongoQueries()
                queries.run_menu()
                queries.close()
            elif choice == '4':
                queries = Neo4jQueries()
                queries.run_menu()
                queries.close()
            elif choice == '5':
                cleaner = SystemCleanup()
                cleaner.run()
            elif choice == '0':
                print("Exiting application.")
                self.running = False
            else:
                print("Invalid choice. Please try again.")

if __name__ == "__main__":
    app = MainMenu()
    app.run()


    CENTRAL KITCHEN DATA PIPELINE
1. Run Full Pipeline (Produce Data & Load DBs)
2. Run Batch Processing (Spark Reports)
3. Run MongoDB Queries
4. Run Neo4j Queries
5. Reset System (Clear Kafka, DBs, HDFS)
0. Exit


Enter your choice (0-5):  1



Starting the automated data pipeline.
Starting Spark Stream Processor...


Process Process-1:
Traceback (most recent call last):
  File "/usr/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/usr/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_10321/4259539343.py", line 16, in run_streamer
    streamer.start_streaming()
  File "/home/student/G5-1/processing/spark_stream_processor.py", line 145, in start_streaming
    kitchen_raw = self.read_kafka_topic(servers, kitchen_topic)
  File "/home/student/G5-1/processing/spark_stream_processor.py", line 77, in read_kafka_topic
    .load()
  File "/home/student/de-venv/lib/python3.10/site-packages/pyspark/sql/streaming/readwriter.py", line 277, in load
    return self._df(self._jreader.load())
  File "/home/student/de-venv/lib/python3.10/site-packages/py4j/java_gateway.py", line 1322, in __call__
    return_value = get_return_value(
  File "/home/student/de-venv/lib/python3.10/site-packages/pyspa

Starting Kafka Consumer...
Connected to Kafka. Listening for events...
(Press Ctrl+C to stop)
Starting Kafka Producer...
Successfully connected to Kafka.
Generator running... Press Ctrl+C to stop.
[kitchen_station_events] BATCH-1001 -> PREPARING | Meta ID: KIT-EVT-1001 | Station: prep_01 | fish_curry | 27.31kg | [{'item': 'white_fish', 'amount_kg': 13.65}, {'item': 'curry_paste', 'amount_kg': 8.19}, {'item': 'coconut_milk', 'amount_kg': 5.46}] | 18.1°C | 2026-03-15T06:01
[kitchen_station_events] BATCH-1003 -> PREPARING | Meta ID: KIT-EVT-1003 | Station: prep_03 | chicken_rice | 46.1kg | [{'item': 'chicken', 'amount_kg': 23.05}, {'item': 'rice', 'amount_kg': 18.44}, {'item': 'water', 'amount_kg': 4.61}] | 15.8°C | 2026-03-15T06:13Limits reached. All batches delivered. Stopping generator.

Closing Kafka connection...[kitchen_station_events] BATCH-1001 -> COOKING | Meta ID: KIT-EVT-1004 | Station: cook_01 | fish_curry | 27.31kg | [{'item': 'white_fish', 'amount_kg': 13.65}, {'item': 'curr

Py4JJavaError: An error occurred while calling None.org.apache.spark.api.java.JavaSparkContext.
: org.apache.spark.SparkException: Only one SparkContext should be running in this JVM (see SPARK-2243).The currently running SparkContext was created at:
org.apache.spark.api.java.JavaSparkContext.<init>(JavaSparkContext.scala:58)
sun.reflect.NativeConstructorAccessorImpl.newInstance0(Native Method)
sun.reflect.NativeConstructorAccessorImpl.newInstance(NativeConstructorAccessorImpl.java:62)
sun.reflect.DelegatingConstructorAccessorImpl.newInstance(DelegatingConstructorAccessorImpl.java:45)
java.lang.reflect.Constructor.newInstance(Constructor.java:423)
py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:247)
py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374)
py4j.Gateway.invoke(Gateway.java:238)
py4j.commands.ConstructorCommand.invokeConstructor(ConstructorCommand.java:80)
py4j.commands.ConstructorCommand.execute(ConstructorCommand.java:69)
py4j.ClientServerConnection.wa